In [1]:
!pip install transformers


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
!pip install pandas torch transformers


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# ===============================
# Load FinBERT ONCE (IMPORTANT)
# ===============================
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

nlp = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    truncation=True,
    max_length=512
)

# ===============================
# FinBERT Sentiment Function
# ===============================
def FinBERT_sentiment_score(news_list):
    scores = []

    for text in news_list:
        if not isinstance(text, str):
            continue
        text = text.strip()
        if text == "" or text == "0":
            continue

        result = nlp(text)
        label = result[0]["label"]
        score = result[0]["score"]

        if label == "positive":
            scores.append(score)
        elif label == "negative":
            scores.append(-score)
        else:
            scores.append(0)

    return np.mean(scores) if scores else 0


# ===============================
# Load Dataset
# ===============================
news_df = pd.read_csv("./STOCKNEPSE/NEPSEDATA/sypnlnews_full_content.csv")

# ===============================
# Apply FinBERT Row-wise
# ===============================
BERT_sentiment = []

for i in range(len(news_df)):
    news_list = news_df.iloc[i, 1:].astype(str).tolist()
    news_list = [n for n in news_list if n != "0"]

    score = FinBERT_sentiment_score(news_list)
    BERT_sentiment.append(score)

# ===============================
# Save Output
# ===============================
news_df["FinBERT score"] = BERT_sentiment
news_df.to_csv("sentiment.csv", index=False)

print("✅ Sentiment analysis completed successfully.")
